# MagiCoder Improved Training Pipeline (Colab)

Three improvements over baseline SFT:
1. **Execution-filtered data** — AST-validates every example; removes broken solutions
2. **Curriculum learning** — Easy -> Medium -> Hard across 3 stages; LR decays each stage
3. **Replay buffer** — 10% of each stage's batch re-samples earlier examples to prevent forgetting

## Expected runtime on Colab A100
| Model | Steps | Est. time |
|-------|-------|-----------|
| 270m | 800x3 = 2400 | ~1.5 h |
| 1b | 1000x3 = 3000 | ~2.5 h |
| 4b (QLoRA) | 1000x3 = 3000 | ~4.5 h |
| **Total** | | **~8-9 h** |

In [ ]:
!pip install -q --upgrade transformers peft trl datasets bitsandbytes accelerate
print("Packages installed.")

In [ ]:
import os, json, ast, re, random, math, gc
import numpy as np
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU:  {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset, load_dataset

_sm = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
DTYPE = torch.bfloat16 if _sm >= 8 else torch.float16
print(f"Compute dtype: {DTYPE}")

In [ ]:
DRIVE_BASE = "/content/drive/MyDrive/honor_project"
SAVE_DIR   = f"{DRIVE_BASE}/improved_lora_outputs"
os.makedirs(SAVE_DIR, exist_ok=True)

# Paste your HuggingFace token (required to download Gemma from Hub).
# OR upload model folders to Drive at DRIVE_BASE/hf_models/{270m,1b,4b}/
HF_TOKEN = ""

# ── Training hyperparameters ──────────────────────────────────────────────────
MAX_SEQ_LENGTH = 512     # packing=True fits ~4-6 examples per sequence
BATCH_SIZE     = 4
GRAD_ACCUM     = 4       # effective batch = 16
LR             = 2e-4
WARMUP_STEPS   = 50
REPLAY_SIZE    = 500     # max examples kept in replay buffer
REPLAY_RATIO   = 0.10    # fraction of each stage drawn from replay buffer

# ── Per-model config ──────────────────────────────────────────────────────────
MODEL_CONFIGS = [
    {"size": "270m", "model_id": "google/gemma-3-270m",
     "lora_r": 32, "lora_alpha": 64, "use_4bit": False, "steps": 800},
    {"size": "1b",   "model_id": "google/gemma-3-1b",
     "lora_r": 32, "lora_alpha": 64, "use_4bit": False, "steps": 1000},
    {"size": "4b",   "model_id": "google/gemma-3-4b",
     "lora_r": 16, "lora_alpha": 32, "use_4bit": True,  "steps": 1000},
]
print(f"Save dir: {SAVE_DIR}")
for _c in MODEL_CONFIGS:
    print(f"  {_c['size']}: {_c['steps']} steps/stage x 3 = {_c['steps']*3} total  "
          f"| {'QLoRA 4-bit' if _c['use_4bit'] else 'bf16/fp16'}")

In [ ]:
def extract_code(text):
    m = re.search(r"```(?:python)?\s*(.*?)\s*```", text, re.DOTALL)
    return m.group(1).strip() if m else text.strip()

def is_valid(example):
    resp = example.get("response") or example.get("solution") or ""
    code = extract_code(resp)
    if not code:
        return False
    try:
        tree = ast.parse(code)
    except SyntaxError:
        return False
    has_def    = any(isinstance(n, ast.FunctionDef)  for n in ast.walk(tree))
    has_return = any(isinstance(n, ast.Return)        for n in ast.walk(tree))
    n_lines    = len([l for l in code.splitlines() if l.strip()])
    return has_def and has_return and 3 <= n_lines <= 60

print("Loading MagiCoder-OSS-Instruct-75K ...")
_raw  = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train",
                     token=HF_TOKEN or None)
_s    = _raw[0]
INSTR_F = "instruction" if "instruction" in _s else "problem"
RESP_F  = "response"    if "response"    in _s else "solution"
print(f"  Field names: instruction='{INSTR_F}'  response='{RESP_F}'")

py_all   = [dict(r) for r in _raw if (r.get("lang") or "").lower() == "python"]
filtered = [r for r in py_all if is_valid(r)]
print(f"  Python total : {len(py_all)}")
print(f"  After filter : {len(filtered)} ({len(filtered)/len(py_all)*100:.1f}% retained)")

In [ ]:
def difficulty(ex):
    code = extract_code(ex.get(RESP_F, "") or "")
    try:
        tree = ast.parse(code)
    except Exception:
        return 1.0
    lines   = len([l for l in code.splitlines() if l.strip()])
    n_loops = sum(1 for n in ast.walk(tree) if isinstance(n, (ast.For, ast.While)))
    n_ifs   = sum(1 for n in ast.walk(tree) if isinstance(n, ast.If))
    def _depth(node, d=0):
        ch = list(ast.iter_child_nodes(node))
        return d if not ch else max(_depth(c, d + 1) for c in ch)
    nest  = min(_depth(tree), 8)
    score = (lines / 60) * 0.35 + (n_loops / 6) * 0.25 + (n_ifs / 6) * 0.20 + (nest / 8) * 0.20
    return min(score, 1.0)

def to_prompt(ex):
    return ("### User:\n" + ex.get(INSTR_F, "")
            + "\n### Assistant:\n" + ex.get(RESP_F, ""))

print("Scoring difficulty (may take ~60 s) ...")
scored        = sorted(filtered, key=difficulty)
n             = len(scored)
easy, medium, hard = scored[:n//3], scored[n//3:2*n//3], scored[2*n//3:]
print(f"  Easy: {len(easy)}  Medium: {len(medium)}  Hard: {len(hard)}")

In [ ]:
class ReplayBuffer:
    def __init__(self, max_size=REPLAY_SIZE):
        self.buf      = []
        self.max_size = max_size

    def add(self, examples, n=200):
        sample = random.sample(examples, min(n, len(examples)))
        self.buf.extend(sample)
        if len(self.buf) > self.max_size:
            self.buf = random.sample(self.buf, self.max_size)

    def mix_into(self, examples):
        n_replay = max(1, int(len(examples) * REPLAY_RATIO))
        replayed = random.sample(self.buf, min(n_replay, len(self.buf))) if self.buf else []
        combined = list(examples) + replayed
        random.shuffle(combined)
        return combined


def load_model(cfg):
    src   = cfg["model_id"]
    local = f"{DRIVE_BASE}/hf_models/{cfg['size']}"
    if os.path.isdir(local):
        src = local
        print(f"  Source: Drive ({local})")
    else:
        print(f"  Source: HF Hub ({src})")
    tok = AutoTokenizer.from_pretrained(src, token=HF_TOKEN or None)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"
    if cfg["use_4bit"]:
        bnb = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=DTYPE, bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            src, quantization_config=bnb, device_map="auto",
            attn_implementation="sdpa", token=HF_TOKEN or None,
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            src, dtype=DTYPE, device_map="auto",
            attn_implementation="sdpa", token=HF_TOKEN or None,
        )
        model.gradient_checkpointing_enable()
    lora = LoraConfig(
        r=cfg["lora_r"], lora_alpha=cfg["lora_alpha"],
        target_modules="all-linear", lora_dropout=0.05,
        bias="none", task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
    return model, tok


def run_stage(model, tok, examples, out_dir, n_steps, lr):
    ds = Dataset.from_dict({"text": [to_prompt(e) for e in examples]})
    args = SFTConfig(
        output_dir=out_dir,
        max_steps=n_steps,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=lr,
        lr_scheduler_type="cosine",
        warmup_steps=min(WARMUP_STEPS, n_steps // 10),
        bf16=(DTYPE == torch.bfloat16),
        fp16=(DTYPE == torch.float16),
        logging_steps=50,
        save_steps=n_steps,
        save_total_limit=1,
        packing=True,
        dataset_text_field="text",
        report_to="none",
    )
    # max_seq_length moved to SFTTrainer in newer TRL versions
    try:
        trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
                             args=args, max_seq_length=MAX_SEQ_LENGTH)
    except TypeError:
        trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds, args=args)
    trainer.train()
    del trainer, ds, args
    gc.collect()

print("Utilities ready.")

In [ ]:
for cfg in MODEL_CONFIGS:
    size = cfg["size"]
    S    = cfg["steps"]
    print(f"\n{'='*65}")
    print(f"Training gemma-3-{size}  |  3 stages x {S} steps = {S*3} total")
    print(f"{'='*65}")

    model, tok = load_model(cfg)
    replay     = ReplayBuffer()
    root       = f"{SAVE_DIR}/gemma-3-{size}"

    # Stage 1: Easy only — full learning rate
    print(f"\n[1/3] Easy  ({len(easy)} examples, lr={LR:.0e})")
    run_stage(model, tok, easy, f"{root}/s1", S, LR)
    replay.add(easy, n=200)
    print(f"  Replay buffer: {len(replay.buf)} examples")

    # Stage 2: Medium + replay — LR halved to preserve stage-1 gains
    lr2 = LR * 0.5
    s2  = replay.mix_into(medium)
    print(f"\n[2/3] Medium + replay  ({len(s2)} examples, lr={lr2:.0e})")
    run_stage(model, tok, s2, f"{root}/s2", S, lr2)
    replay.add(medium, n=150)
    print(f"  Replay buffer: {len(replay.buf)} examples")

    # Stage 3: Hard + replay — LR quartered to preserve earlier learning
    lr3 = LR * 0.25
    s3  = replay.mix_into(hard)
    print(f"\n[3/3] Hard + replay  ({len(s3)} examples, lr={lr3:.0e})")
    run_stage(model, tok, s3, f"{root}/s3", S, lr3)

    # Save final LoRA adapter to Drive
    final = f"{root}/final_lora"
    model.save_pretrained(final)
    tok.save_pretrained(final)
    print(f"\n  Saved: {final}")

    del model, tok, replay
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll models trained. Adapters saved to Drive.")